In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib
import os
print("✅ Imports done!")

In [ ]:
df = pd.read_csv("../data/india_housing_prices.csv")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

In [ ]:
print("=== Data Types ===")
print(df.dtypes)
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Basic Stats ===")
df.describe()

In [ ]:
df = df.drop(columns=['ID'])
print("Dropped ID. New shape:", df.shape)

In [ ]:
# ── Basic derived features ──────────────────────────────────
df['Amenity_Count']     = df['Amenities'].str.count(',') + 1
df['floor_ratio']       = df['Floor_No'] / (df['Total_Floors'] + 1)
df['location_score']    = df['Nearby_Schools'] + df['Nearby_Hospitals']
df['size_per_bhk']      = df['Size_in_SqFt'] / df['BHK']
df['total_value_score'] = df['location_score'] * df['Amenity_Count']
df['is_new_property']   = (df['Age_of_Property'] <= 5).astype(int)
df['is_top_floor']      = (df['Floor_No'] == df['Total_Floors']).astype(int)
df['bhk_size_ratio']    = df['BHK'] / (df['Size_in_SqFt'] / 1000)

# ── City/State level aggregate features ─────────────────────
city_med_psqft   = df.groupby('City')['Price_per_SqFt'].transform('median')
city_med_schools = df.groupby('City')['Nearby_Schools'].transform('median')
state_med_psqft  = df.groupby('State')['Price_per_SqFt'].transform('median')

df['city_price_ratio']  = df['Price_per_SqFt'] / (city_med_psqft + 1e-5)
df['below_city_median'] = (df['Price_per_SqFt'] < city_med_psqft).astype(int)
df['city_school_ratio'] = df['Nearby_Schools'] / (city_med_schools + 1)

print("✅ Feature engineering done!")
print("New shape:", df.shape)
print("New features added:", ['Amenity_Count','floor_ratio','location_score',
    'size_per_bhk','total_value_score','is_new_property','is_top_floor',
    'bhk_size_ratio','city_price_ratio','below_city_median','city_school_ratio'])

In [ ]:
# ── Regression Target: Price_per_SqFt ───────────────────────
# Price_per_SqFt has real correlation (0.55) with Price_in_Lakhs
# Model learns: given property features → what price/sqft should it be?
df['Future_Price'] = df['Price_per_SqFt']

# ── Classification Target: Good Investment ───────────────────
# Rule: property is a good investment if:
#   1. Price per sqft is BELOW city median (good value)
#   2. BHK >= 2 (decent size)
#   3. Location score (schools+hospitals) >= 8 (good amenities)
city_med_psqft = df.groupby('City')['Price_per_SqFt'].transform('median')

df['Good_Investment'] = (
    (df['Price_per_SqFt'] <= city_med_psqft) &
    (df['BHK'] >= 2) &
    (df['location_score'] >= 8)
).astype(int)

print("✅ Targets created!")
print("\nGood_Investment distribution:")
print(df['Good_Investment'].value_counts())
print(f"Positive ratio: {df['Good_Investment'].mean():.2%}")
print("\nFuture_Price (Price_per_SqFt) stats:")
print(df['Future_Price'].describe())

In [ ]:
categorical_cols = [
    'State', 'City', 'Locality', 'Property_Type',
    'Furnished_Status', 'Public_Transport_Accessibility',
    'Parking_Space', 'Security', 'Amenities',
    'Facing', 'Owner_Type', 'Availability_Status'
]

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"✅ Encoded: {col}")

print("\nAll categorical columns encoded!")

In [ ]:
numeric_cols = [
    'BHK', 'Size_in_SqFt', 'Year_Built', 'Floor_No', 'Total_Floors',
    'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals',
    'Amenity_Count', 'floor_ratio', 'location_score', 'size_per_bhk',
    'total_value_score', 'bhk_size_ratio', 'city_school_ratio'
]

scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
print("✅ Numeric columns scaled!")
df[numeric_cols].describe().round(2)

In [ ]:
feature_cols = [
    'State','City','Locality','Property_Type','BHK','Size_in_SqFt',
    'Year_Built','Furnished_Status','Floor_No','Total_Floors','Age_of_Property',
    'Nearby_Schools','Nearby_Hospitals','Public_Transport_Accessibility',
    'Parking_Space','Security','Amenities','Facing','Owner_Type','Availability_Status',
    'Amenity_Count','floor_ratio','location_score','size_per_bhk',
    'total_value_score','is_new_property','is_top_floor','bhk_size_ratio',
    'below_city_median','city_school_ratio'
]

joblib.dump(feature_cols,   "../models/feature_columns.pkl")
joblib.dump(feature_cols,   "../models/clf_feature_columns.pkl")
joblib.dump(feature_cols,   "../models/reg_feature_columns.pkl")
joblib.dump(label_encoders, "../models/label_encoders.pkl")
joblib.dump(scaler,         "../models/scaler.pkl")

print("✅ feature_columns.pkl saved")
print("✅ clf_feature_columns.pkl saved")
print("✅ reg_feature_columns.pkl saved")
print("✅ label_encoders.pkl saved")
print("✅ scaler.pkl saved")
print(f"\nTotal features: {len(feature_cols)}")

In [ ]:
df.to_csv("../data/cleaned_data.csv", index=False)
print("✅ Cleaned dataset saved!")
print("Final shape:", df.shape)
df.head()